# SE Australia Observed Fire-Season Amplification (ERA5)

Computes the **fire-season warming response** β = (regional fire-season trend) / (global
GMST trend) from ERA5 — the physically meaningful coefficient used as the primary additive
shift in the Black Summer GEV shift-fit (notebook 04).

> **Methodology note.** A previous version of this notebook multiplied the Probability Ratio
> by an amplification *ratio* (`PR_obs = PR_era5 × α_obs/α_cmip6`). That is statistically
> invalid — PR is a nonlinear function of the distributional shift and does not scale
> linearly with an amplification ratio. The amplification factor enters correctly as the
> **shift coefficient β** inside the shift-fit (notebook 04), where β=0.726 is primary and
> β=0.935 is a sensitivity. The invalid `liability_obs_*` columns have been removed.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('../../').resolve()
sys.path.insert(0, str(ROOT))
from src.attribution import (
    area_weighted_series, season_block_max, wet_season_max_ndays,
    load_gmst, extrapolate_to, smoothed_covariate, event_gmst_sigma,
    shift_fit_gev, fit_gev, build_liability_table, far,
    AUD_TO_USD, CC_RATE_STANDARD, CC_RATE_HIGH, CLIM_START, CLIM_END,
)
from scipy.stats import genextreme

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
RAW  = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
FIGS = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
print('Setup complete.')


In [ ]:
LAT_S, LAT_N = -44, -28
LON_W, LON_E = 138, 154
FIRE_MONTHS  = [10, 11, 12, 1, 2, 3]
TREND_START, TREND_END = 1961, 2020
from scipy.stats import linregress

ds = xr.open_dataset(RAW / 'era5' / 'era5_mx2t_daily_se_australia_1961_2020.nc')
ts_daily = area_weighted_series(ds['mx2t'])
if ts_daily.mean() > 100:
    ts_daily = ts_daily - 273.15

# Fire-season MEAN of daily max (trend metric, not block max)
ts_fs = ts_daily[ts_daily.index.month.isin(FIRE_MONTHS)].copy()
ts_fs.index = ts_fs.index - pd.DateOffset(months=9)
fire_season_mean = ts_fs.resample('YE').mean().dropna()
fire_season_mean.index = fire_season_mean.index.year
fs = fire_season_mean.loc[TREND_START:TREND_END]

slope_au = linregress(fs.index, fs.values).slope
ft = load_gmst(PROC)['t_p50'].loc[TREND_START:TREND_END]
slope_gl = linregress(ft.index, ft.values).slope
obs_amp = slope_au / slope_gl

print(f'ERA5 SE AU fire-season mean mx2t trend: {slope_au*10:.4f} °C/decade')
print(f'FaIR GMST trend:                        {slope_gl*10:.4f} °C/decade')
print(f'ERA5 observed fire-season amplification β = {obs_amp:.3f}')


In [ ]:
# Persist (recomputed; matches prior value). Used as primary β in notebook 04.
out = pd.DataFrame([{
    'source': 'ERA5_observed',
    'trend_global_per_yr': slope_gl,
    'trend_au_per_yr': slope_au,
    'amplification': obs_amp,
    'period': f'{TREND_START}-{TREND_END}',
    'metric': 'fire_season_mean_mx2t',
}])
out.to_csv(PROC / 'observed_amplification_factor.csv', index=False)
print('Saved observed_amplification_factor.csv')
print(out.to_string(index=False))

print('\nComparison of amplification estimates (all enter as the shift coefficient β):')
print(f'  ERA5 observed (fire-season mx2t):     {obs_amp:.3f}   ← PRIMARY')
au = pd.read_csv(PROC / 'au_amplification_factor.csv')
print(f'  CMIP6 ensemble (ANNUAL-mean tas):     {au["amplification"].median():.3f}   ← sensitivity')
print('  NOTE: the CMIP6 value is an annual-mean tas amplification (notebook 02),')
print('  a different metric from the ERA5 fire-season value — they are not directly')
print('  comparable, which is why the fire-season ERA5 estimate is used as primary.')


In [ ]:
# ── β-sensitivity of the Black Summer PR (the statistically valid use of α) ──
gmst = load_gmst(PROC)
covariate = smoothed_covariate(gmst['t_p50'])
g_sigma = event_gmst_sigma(gmst, 2019)
anom_fs = season_block_max(ts_daily.resample('ME').mean(), FIRE_MONTHS)
anom = anom_fs - anom_fs.loc[CLIM_START:CLIM_END].mean()

print('Black Summer PR as a function of the shift coefficient β:')
for label, b in [('ERA5 obs β=0.726 (PRIMARY)', obs_amp),
                 ('CMIP6 tas β=0.935', float(au['amplification'].median()))]:
    r = shift_fit_gev(anom, covariate, 2019, mode='additive', beta=b, g_event_sigma=g_sigma)
    print(f'  {label:28s}: PR={r.pr:.2f} [{r.pr_p05:.2f}–{r.pr_p95:.2f}]  FAR={r.far:.3f}')
print('\nHigher β → larger counterfactual shift → higher PR. This is the correct,')
print('nonlinear dependence — not a linear PR×ratio scaling.')
